# 01 - Intestinal epithelial reporter expression

In [ ]:
%autosave 0

In [ ]:
import os
import sys
import dill
from pathlib import Path

# Add sc-ops to path
sys.path.append('/projects/site/pred/ihb-g-deco/USERS/adaml9/sc-ops')

from functools import partial
from pathlib import Path
from sc_ops.settings import *
from sc_ops.preprocessing._adata_ops import *
from sc_ops.preprocessing._tau_score import tau_score

## Load data

In [ ]:
# Manually set the external data directory path
external_data_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/external/gut_cell_atlas")

# Load the reference atlas from the external data directory
adata = sc.read(external_data_dir / "epi_log_counts02_v2.h5ad")

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color="annotation")

## Define reporters

In [ ]:
marker_dict = {
    "Stem": ["RGMB", "LGR5", "OLFM4", "FGFBP1"],
    "Proliferation": ["PCNA", "MKI67"],
    "Absorptive": ["FABP1", "CYP3A4", "SI", "PIGR", "APOA4", 
                    "AQP10", "CEACAM7", "KRT20", "BEST4", 
                    "CA7"],
    "Progenitors": ["ATOH1", "NEUROG3"],
    "Goblet": ["ITLN1", "SPINK4", "ZG16", "MUC2", "FCGBP"],
    "Paneth": ["DEFA5", "PLA2G2A"],
    "EEC": ["CHGA", "VWA5B2", "SST", "HHEX", "GIP", "GAST", 
            "GCG", "PYY", "NTS", "INSL5", "CCK", "ONECUT3", 
            "TPH1", "LMX1A", "MLN", "GHRL", "SCT"],
    "M": ["GP2", "SPIB"],
    "Tuft cell": ["DCLK2", "AVIL", "TRPM5"],
}

In [ ]:
# Convert to long-format DataFrame
df = (
    pd.DataFrame(
        [(annotation, gene) for annotation, genes in marker_dict.items() for gene in genes],
        columns=["annotation", "gene"]
    )
)

In [ ]:
# Select genes that are present in the dataset
genes_in_data = [gene for gene in df["gene"] if gene in adata.var_names]
print(f"Number of genes in marker list: {len(df)}")
print(f"Number of genes present in dataset: {len(genes_in_data)}")

# Give me the genes that are in the marker list but not in the dataset
genes_not_in_data = [gene for gene in df["gene"] if gene not in adata.var_names]
print(f"Genes in marker list but not in dataset: {genes_not_in_data}")

In [ ]:
adata_subset = adata[:, genes_in_data].copy()
adata_subset = adata_subset[adata_subset.obs["Diagnosis"].isin(["Healthy adult"])]
adata_subset = adata_subset[adata_subset.obs["Age_group"].isin(["Adult"])]

# Remove all cells annotated as EECs under the "annotation" column
adata_subset = adata_subset[~adata_subset.obs["annotation"].str.contains("EECs")]

In [ ]:
# Group Enterocyte, Colonocyte, BEST4+ epithelial under "Absorptive" annotation
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"Enterocyte": "Absorptive", "Colonocyte": "Absorptive", "BEST4+ epithelial": "Absorptive"}
)

# Group BEST2+ Goblet cell & Goblet cell under "Goblet" annotation
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"BEST2+ Goblet cell": "Goblet", "Goblet cell": "Goblet"}
)

# Rename Microfold cell as "M" annotation
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"Microfold cell": "M"}
)
# Rename Stem cells as "Stem" annotation
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"Stem cells": "Stem"}
)

# Rename L cells (PYY+) as "L" annotation
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"L cells (PYY+)": "L"}
)

# Same for I cells (CCK+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"I cells (CCK+)": "I/N"}
)

# Same for K cells (GIP+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"K cells (GIP+)": "K"}
)

# Same for D cells (SST+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"D cells (SST+)": "D"}
)

# Same for EC cells (TAC1+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"EC cells (TAC1+)": "EC"}
)

# Same for M/X cells (MLN/GHRL+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"M/X cells (MLN/GHRL+)": "X"}     
)

# Same for N cells (NTS+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"N cells (NTS+)": "I/N"}
)

# Same for Progenitor (NEUROG3+)
adata_subset.obs["annotation"] = adata_subset.obs["annotation"].replace(
    {"Progenitor (NEUROG3+)": "Progenitor"}
)

In [ ]:
# Count number of cells per batch
cell_counts = adata_subset.obs["batch"].value_counts()

# Remove batches with less than 50 cells
batches_to_keep = cell_counts[cell_counts >= 50].index

# Print how many batches are filtered out and how many are kept
print(f"Number of batches before filtering: {len(cell_counts)}")
print(f"Number of batches after filtering: {len(batches_to_keep)}")

In [ ]:
# Subset df to genes in the dataset
df_subset = df[df["gene"].isin(genes_in_data)]

# Compute average expression per annotation group & normalize per gene to be between 0 and 1
mean_df = aggregate(adata_subset, groupby="annotation", method="mean", normalize="minmax", normalize_axis="gene")

# Reorder df_subset based on the order of the columns in mean_df
df_subset["gene"] = pd.Categorical(df_subset["gene"], categories=mean_df.columns, ordered=True)
df_subset = df_subset.sort_values("gene")

In [ ]:
import marsilea as ma
import marsilea.plotter as mp

h = ma.Heatmap(mean_df, cmap="Greys", width=10, label="Mean expression\n(normalized)")
h.add_bottom(mp.Labels(mean_df.columns, rotation=90))
h.add_left(mp.Labels(mean_df.index, rotation=0))
h.add_top(mp.Bar(tau_score(mean_df.T).values, label="SP"), size=0.3, pad=0.1)
h.add_top(mp.Colors(df_subset["annotation"]), size=0.1, pad=0.1, legend=False)
h.add_top(mp.Labels(df_subset["annotation"], rotation=90), pad=0.1)
h.add_legends()
h.render()

plt.savefig("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/reporter_expression_heatmap_v2.pdf", dpi=300, bbox_inches="tight")